# Importar librerías y datos

In [38]:
from pathlib import Path
import pandas as pd

In [39]:
RUTA_DATOS = Path("data") / "KaggleV2-May-2016.csv"

assert RUTA_DATOS.exists(), f"No se encontró el archivo: {RUTA_DATOS}"

df = pd.read_csv(RUTA_DATOS)

print("Dataset cargado.")
print("Dimensiones", df.shape)

Dataset cargado.
Dimensiones (110527, 14)


# Limpieza de datos

* Recordamos que no había datos nulos
* Recordamos que había al menos una edad negativa
* Recordamos que habian registros con la cita médica antes que el registro

\* Todo esto visto en el perfilamiento de calidad del dataset.

In [40]:
edades_negativas = df[df['Age'] < 0]
print(f"\nCantidad de registros con edad negativa: {len(edades_negativas)}")
df = df[df['Age'] >= 0]

# Convertimos a datetime temporalmente para poder usar .dt.date si las columnas aún son strings
fecha_agendado = pd.to_datetime(df['ScheduledDay']).dt.date
fecha_cita = pd.to_datetime(df['AppointmentDay']).dt.date

inconsistencias_fechas = df[fecha_cita < fecha_agendado]
print(f"Cantidad de registros donde la cita es anterior al registro: {len(inconsistencias_fechas)}")

df = df[fecha_cita >= fecha_agendado]

print("\n--- Dataset limpio ---")
print("Dimensiones finales:", df.shape)


Cantidad de registros con edad negativa: 1
Cantidad de registros donde la cita es anterior al registro: 5

--- Dataset limpio ---
Dimensiones finales: (110521, 14)


# Corregir tipos de datos

Primero limpiamos los datos anteriormente porque es una buena práctica hacer eso antes de hacer cambios de tipo de dato, especialmente en datasets enormes. Ahora procedemos a corregir los tipos de datos.

In [41]:
# Mostrar los tipos de datos iniciales
print("--- Tipos de datos iniciales ---")
print(df.dtypes)

# 1. Convertir IDs
# PatientId viene como float, lo ideal es que sea int64
df['PatientId'] = df['PatientId'].astype('int64')

# 2. Convertir columnas de fechas a datetime (corregido ScheduledDate -> ScheduledDay y AppointmentDate -> AppointmentDay)
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

# 3. Convertir variables binarias/indicadores a booleanos para optimizar espacio y legibilidad
columnas_booleanas = ['Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'SMS_received', 'No-show']
df['No-show'] = df['No-show'].map({'Yes': True, 'No': False})
for col in columnas_booleanas:
    df[col] = df[col].astype(bool)

# 4. 'Handcap' contiene valores de 0 a 4 (número de discapacidades), lo dejamos como entero
df['Handcap'] = df['Handcap'].astype('int32')

# 5. Convertir el objetivo 'Gender' a categoría
df['Gender'] = df['Gender'].astype('category')

# Mostrar los tipos de datos corregidos
print("\n--- Tipos de datos corregidos ---")
print(df.dtypes)

# Mostrar una vista previa de los datos
df.head()

--- Tipos de datos iniciales ---
PatientId         float64
AppointmentID       int64
Gender             object
ScheduledDay       object
AppointmentDay     object
Age                 int64
Neighbourhood      object
Scholarship         int64
Hipertension        int64
Diabetes            int64
Alcoholism          int64
Handcap             int64
SMS_received        int64
No-show            object
dtype: object

--- Tipos de datos corregidos ---
PatientId                       int64
AppointmentID                   int64
Gender                       category
ScheduledDay      datetime64[ns, UTC]
AppointmentDay    datetime64[ns, UTC]
Age                             int64
Neighbourhood                  object
Scholarship                      bool
Hipertension                     bool
Diabetes                         bool
Alcoholism                       bool
Handcap                         int32
SMS_received                     bool
No-show                          bool
dtype: object


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,29872499824296,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,False,True,False,False,0,False,False
1,558997776694438,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,False,False,False,0,False,False
2,4262962299951,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,False,False,False,False,0,False,False
3,867951213174,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,False,False,False,False,0,False,False
4,8841186448183,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,True,True,False,0,False,False


# Ingeniería de variables

In [42]:
# 1. Calcular el tiempo de espera en días (WaitingDays) utilizando las columnas de fecha originales de tipo datetime
df['WaitingDays'] = (df['AppointmentDay'].dt.normalize() - df['ScheduledDay'].dt.normalize()).dt.days

# 2. Obtener el día de la semana de la cita
df['AppointmentDayOfWeek'] = df['AppointmentDay'].dt.day_name().astype('category')

# 3. Sumar condiciones médicas (comorbilidades)
# Convertimos los booleanos a enteros (0 o 1) para la suma
df['Num_Conditions'] = (
    df['Hipertension'].astype(int) +
    df['Diabetes'].astype(int) +
    df['Alcoholism'].astype(int) +
    (df['Handcap'] > 0).astype(int) # Consideramos si tiene al menos una discapacidad
)

# Mostrar vista previa con las nuevas variables creadas
print("--- Nuevas variables creadas con éxito ---")
display(df[['ScheduledDay', 'AppointmentDay', 'WaitingDays', 'AppointmentDayOfWeek', 'Num_Conditions']].head())
print("\nDimensiones actuales del dataset:", df.shape)

--- Nuevas variables creadas con éxito ---


,ScheduledDay,AppointmentDay,WaitingDays,AppointmentDayOfWeek,Num_Conditions
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,0,Friday,1
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,0,Friday,0
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,0,Friday,0
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,0,Friday,0
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,0,Friday,2



Dimensiones actuales del dataset: (110521, 17)



# Anonimización y Seguridad de Datos (Privacidad)

### Justificación
Al trabajar con datos relacionados con la salud (citas médicas), es fundamental proteger la privacidad de los pacientes y cumplir con estándares de seguridad de datos (como HIPAA o GDPR), incluso si el conjunto de datos es de uso académico o de desarrollo interno:
1. **Eliminación de Identificadores Innecesarios:** `AppointmentID` es un identificador único transaccional que no aporta valor predictivo para un modelo de Machine Learning y expone una clave de registro directo. Por ello, se elimina por completo.
2. **Pseudo-anonimización de IDs de Paciente:** `PatientId` contiene identificadores reales que podrían ligar los registros a individuos si se combinaran con otras bases de datos externas. Sin embargo, no podemos eliminarlo por completo porque la recurrencia de un paciente (si asiste o falta repetidamente) es una característica de gran valor predictivo. En su lugar, aplicamos **pseudo-anonimización**: mapeamos de forma consistente cada `PatientId` original a un identificador ficticio secuencial (un ID mapeado de manera única), de modo que preservamos el patrón de comportamiento de cada paciente sin exponer su ID real.


In [43]:
# 1. Crear un mapeo único para pseudo-anonimizar PatientId de forma consistente
unique_patients = df['PatientId'].unique()
patient_id_map = {original_id: f"Paciente_{i+1}" for i, original_id in enumerate(unique_patients)}

# Aplicar el mapeo al DataFrame
df['PatientId'] = df['PatientId'].map(patient_id_map).astype('category')

# 2. Eliminar el identificador directo AppointmentID
if 'AppointmentID' in df.columns:
    df = df.drop(columns=['AppointmentID'])

# Mostrar vista previa de los datos anonimizados
print("--- Dataset Anonimizado con Éxito ---")
display(df[['PatientId', 'Gender', 'Age', 'Neighbourhood', 'No-show']].head())

--- Dataset Anonimizado con Éxito ---


,PatientId,Gender,Age,Neighbourhood,No-show
0,Paciente_1,F,62,JARDIM DA PENHA,False
1,Paciente_2,M,56,JARDIM DA PENHA,False
2,Paciente_3,F,62,MATA DA PRAIA,False
3,Paciente_4,F,8,PONTAL DE CAMBURI,False
4,Paciente_5,F,56,JARDIM DA PENHA,False


# Dataset final preprocesado

In [44]:
df.head()

,PatientId,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,WaitingDays,AppointmentDayOfWeek,Num_Conditions
0,Paciente_1,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,False,True,False,False,0,False,False,0,Friday,1
1,Paciente_2,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,False,False,False,0,False,False,0,Friday,0
2,Paciente_3,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,False,False,False,False,0,False,False,0,Friday,0
3,Paciente_4,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,False,False,False,False,0,False,False,0,Friday,0
4,Paciente_5,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,True,True,False,0,False,False,0,Friday,2


In [45]:
print(df['No-show'].sum())
print(df.shape)

22314
(110521, 16)


In [46]:
from google.colab import files

# index=False evita que se agregue una columna extra con los números de fila (0, 1, 2...)
df.to_csv("citas_medicas_preprocesadas.csv", index=False)

# 2. Descargar el archivo a tu PC
files.download("citas_medicas_preprocesadas.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

\* Se debe tomar en cuenta que los archivos csv no guardan tipos de datos, asi que será necesario en otros notebooks el cambiar el tipo de las variables ScheduledDay, AppointmentDay a fecha nuevamente.